In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install catboost vmdpy -q

import sys, importlib, json, pickle, time, os
import numpy as np, pandas as pd
sys.path.insert(0,'/content/drive/MyDrive/RuzgarTahmin')
import rt_core; importlib.reload(rt_core); from rt_core import *
import tensorflow as tf
from catboost import CatBoostRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from vmdpy import VMD

OUT = '/content/drive/MyDrive/RuzgarTahmin/yeni/'
SEED = 42; H_LIST = [1,2,4,6,8,12]; SEQ_OPTS = [4,8,12]; PURGE = 12; L_BEST = 4
ALPHA, TAU, K, DC, INIT, TOL = 2000, 0.0, 4, 0, 1, 1e-7
W = 672

# veri ve protokol
df_all = pd.read_csv('/content/drive/MyDrive/RuzgarTahmin/veri/sdwpf_2001_2112_full.csv')
d = add_features(load_turbine(df_all, 57))
tr, va, te = chrono_split(d, purge=PURGE)
mn = tr[FEATURES].min().values; mx = tr[FEATURES].max().values
rng_ = np.where(mx-mn==0, 1, mx-mn)
sc = lambda x: (x[FEATURES].values-mn)/rng_
x_dec = d['Patv'].ffill().fillna(0).values.astype(float); N = len(x_dec)

def vmd_sorted(sig):
    u, _, om = VMD(sig, ALPHA, TAU, K, DC, INIT, TOL)
    return u[np.argsort(om[-1])]
def last_mode(sig): return vmd_sorted(sig)[:, -1]

# yardımcı fonksiyonlar
def build_lstm(L, F):
    m = Sequential([Input(shape=(L, F)), LSTM(64, return_sequences=True), Dropout(0.2),
                    LSTM(32), Dropout(0.2), Dense(1)])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse'); return m
def fit_lstm(D, L):
    tf.keras.utils.set_random_seed(SEED)
    m = build_lstm(L, D['tr']['X'].shape[2])
    m.fit(D['tr']['X'], D['tr']['y']/KAPASITE, validation_data=(D['va']['X'], D['va']['y']/KAPASITE),
          epochs=50, batch_size=256, verbose=0, callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])
    return m
def pred_lstm(m, X): return np.clip(m.predict(X, verbose=0).flatten()*KAPASITE, 0, None)
def scale_comp(U):
    pos = d.index.get_indexer(tr.index)
    lo, hi = np.nanmin(U[pos],0), np.nanmax(U[pos],0)
    return (U-lo)/np.where(hi-lo==0, 1, hi-lo)
def get_xy_ext(L, H, U):
    out = {}
    for ad, x in (('tr',tr),('va',va),('te',te)):
        idx = d.index.get_indexer(x.index)
        Xw, t, y, yc = make_windows(np.hstack([sc(x), U[idx]]), x['Patv'].values, L, H)
        out[ad] = dict(X=Xw, t=t, y=y, yc=yc)
    return out

# kayıtlı sonuçlar ve bileşenler
results = pickle.load(open(OUT+'adim2_T57_tahminler.pkl','rb'))
uA = np.load(OUT+'vmd_full_T57.npy'); uB = np.load(OUT+'vmd_roll_T57.npy')
uA_m = uA.copy(); uA_m[:W-1] = np.nan
Us = {'B6': scale_comp(uA_m), 'B7': scale_comp(uB)}

print('Kayıtlı sonuç anahtarları:', sorted({k[0] for k in results}))
print('Satır sayısı N =', N, '| uA:', uA.shape, '| uB:', uB.shape)
print('Bölme:', tr.index.min().date(), '→', te.index.max().date())

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00
Kayıtlı sonuç anahtarları: ['B1', 'B2', 'B3', 'B6', 'B7']
Satır sayısı N = 35089 | uA: (35089, 4) | uB: (35089, 4)
Bölme: 2020-01-01 → 2022-01-01


In [ ]:
from scipy.signal import butter, lfilter, lfilter_zi, filtfilt

# 1) Bantları YALNIZCA eğitim bölümünden öğren ve kilitle
x_tr_seg = x_dec[:len(tr)]
_, _, om_tr = VMD(x_tr_seg, ALPHA, TAU, K, DC, INIT, TOL)
fc = np.sort(om_tr[-1])
print('Eğitimden öğrenilen merkez frekanslar (döngü/adım):', fc.round(5))
fcl = np.clip(fc, 1e-4, 0.49)
edges = np.sqrt(fcl[:-1]*fcl[1:])                 # K-1 sınır (geometrik orta)
print('Bant sınırları:', edges.round(5))

bands = []
for k in range(K):
    if k == 0:       ba = butter(3, edges[0], 'low', fs=1.0)
    elif k == K-1:   ba = butter(3, edges[K-2], 'high', fs=1.0)
    else:            ba = butter(3, [edges[k-1], edges[k]], 'band', fs=1.0)
    bands.append(ba)

def make_lcsf(x, causal=True):
    out = np.zeros((len(x), K))
    for k, (b, a) in enumerate(bands):
        if causal:
            zi = lfilter_zi(b, a)*x[0]; out[:,k] = lfilter(b, a, x, zi=zi)[0]
        else:
            out[:,k] = filtfilt(b, a, x)           # çift yönlü → geleceğe bakar (SIZDIRAN)
    return out

cC = make_lcsf(x_dec, True); cL = make_lcsf(x_dec, False)
cC[:W-1] = np.nan; cL[:W-1] = np.nan              # B6/B7 ile aynı örnek kümesi
Us['B8']  = scale_comp(cC); Us['B8L'] = scale_comp(cL)

# nedensellik kanıtı
x2 = x_dec.copy(); x2[20001:] += np.random.default_rng(0).normal(0, 200, N-20001)
print('Nedensel LCSF fark (gelecek bozulunca):', np.abs(make_lcsf(x_dec,True)[20000]-make_lcsf(x2,True)[20000]).max(), '[beklenen 0]')
print('Çift yönlü LCSF fark                  :', np.abs(make_lcsf(x_dec,False)[20000]-make_lcsf(x2,False)[20000]).max().round(2), '[beklenen >0 → sızıntı]')

Eğitimden öğrenilen merkez frekanslar (döngü/adım): [0.00054 0.01849 0.04388 0.09026]
Bant sınırları: [0.00316 0.02848 0.06293]
Nedensel LCSF fark (gelecek bozulunca): 0.0 [beklenen 0]
Çift yönlü LCSF fark                  : 33.79 [beklenen >0 → sızıntı]


In [ ]:
H4 = [2, 6, 12]                    # 1 sa, 3 sa, 6 sa
base_models, Dcache = {}, {}
rows = []
for H in H4:
    for key, ad in (('B8','B8 LCSF–LSTM (nedensel)'), ('B8L','B8L LCSF–LSTM (sızdıran)')):
        D = get_xy_ext(L_BEST, H, Us[key]); te_ = D['te']
        assert np.array_equal(te_['t'], results[('B1',H)]['t'])
        m = fit_lstm(D, L_BEST); p = pred_lstm(m, te_['X'])
        results[(key,H)] = dict(t=te_['t'], y=te_['y'], yc=te_['yc'], p=p)
        rows.append(dict(model=ad, H_saat=H/2, **metrics(te_['y'], p, te_['yc'])))
        if key == 'B8': base_models[H] = m; Dcache[H] = D
    print(f'H={H/2} sa tamam')
print(pd.DataFrame(rows).round(3).to_string(index=False))
print('\nLCSF için LSI (aynı aile: sızdıran vs nedensel):')
for H in H4:
    ml = np.mean(np.abs(results[('B8L',H)]['y']-results[('B8L',H)]['p']))
    mc = np.mean(np.abs(results[('B8',H)]['y']-results[('B8',H)]['p']))
    print(f'  H={H/2} sa: leaky={ml:.2f} causal={mc:.2f} LSI={lsi(mc,ml):.2f}%')
pickle.dump(results, open(OUT+'adim2_T57_tahminler.pkl','wb')); print('✓ Kaydedildi')

H=1.0 sa tamam
H=3.0 sa tamam
H=6.0 sa tamam
                   model  H_saat     MAE    RMSE   nMAE  nRMSE    R2  Skill
 B8 LCSF–LSTM (nedensel)     1.0 116.291 181.064  7.753 12.071 0.805 -7.265
B8L LCSF–LSTM (sızdıran)     1.0  57.340  95.511  3.823  6.367 0.946 47.110
 B8 LCSF–LSTM (nedensel)     3.0 189.590 264.907 12.639 17.660 0.580 -0.903
B8L LCSF–LSTM (sızdıran)     3.0  91.950 135.201  6.130  9.013 0.891 51.063
 B8 LCSF–LSTM (nedensel)     6.0 243.518 327.642 16.235 21.843 0.357  4.701
B8L LCSF–LSTM (sızdıran)     6.0 116.059 160.244  7.737 10.683 0.846 54.581

LCSF için LSI (aynı aile: sızdıran vs nedensel):
  H=1.0 sa: leaky=57.34 causal=116.29 LSI=50.69%
  H=3.0 sa: leaky=91.95 causal=189.59 LSI=51.50%
  H=6.0 sa: leaky=116.06 causal=243.52 LSI=52.34%
✓ Kaydedildi


In [ ]:
def fit_arrays(Xt, yt, Xv, yv, L):
    tf.keras.utils.set_random_seed(SEED)
    m = build_lstm(L, Xt.shape[2])
    m.fit(Xt, yt/KAPASITE, validation_data=(Xv, yv/KAPASITE), epochs=50, batch_size=256, verbose=0,
          callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])
    return m

def oof_predictions(Dtr, L, H, n_blocks=4):
    n = len(Dtr['y']); ed = np.linspace(0, n, n_blocks+1).astype(int)
    oof = np.full(n, np.nan); gap = H + L
    for j in range(1, n_blocks):
        te_end = ed[j] - gap                         # eğitim ile tahmin bloğu arasında boşluk
        cut = int(te_end*0.85)                       # erken durdurma: zaman sıralı son %15
        m = fit_arrays(Dtr['X'][:cut], Dtr['y'][:cut], Dtr['X'][cut:te_end], Dtr['y'][cut:te_end], L)
        oof[ed[j]:ed[j+1]] = pred_lstm(m, Dtr['X'][ed[j]:ed[j+1]])
    return oof

rows = []
for H in H4:
    D = Dcache[H]; base = base_models[H]
    oof = oof_predictions(D['tr'], L_BEST, H); ok = ~np.isnan(oof)
    F = lambda X, p: np.column_stack([X[:,-1,:], p])
    base_va = pred_lstm(base, D['va']['X']); base_te = pred_lstm(base, D['te']['X'])
    cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, loss_function='MAE',
                           random_seed=SEED, verbose=0, early_stopping_rounds=50)
    cb.fit(F(D['tr']['X'][ok], oof[ok]), D['tr']['y'][ok]-oof[ok],
           eval_set=(F(D['va']['X'], base_va), D['va']['y']-base_va))
    p9 = np.clip(base_te + cb.predict(F(D['te']['X'], base_te)), 0, None)
    results[('B9',H)] = dict(t=D['te']['t'], y=D['te']['y'], yc=D['te']['yc'], p=p9)
    rows.append(dict(model='B9 LCSF–LSTM + OOF CatBoost', H_saat=H/2, OOF_örnek=int(ok.sum()),
                     **metrics(D['te']['y'], p9, D['te']['yc'])))
    print(f'H={H/2} sa tamam')
print(pd.DataFrame(rows).round(3).to_string(index=False))
pickle.dump(results, open(OUT+'adim2_T57_tahminler.pkl','wb')); print('✓ Kaydedildi')

H=1.0 sa tamam
H=3.0 sa tamam
H=6.0 sa tamam
                      model  H_saat  OOF_örnek     MAE    RMSE   nMAE  nRMSE    R2  Skill
B9 LCSF–LSTM + OOF CatBoost     1.0      15678 108.531 185.200  7.235 12.347 0.796 -0.107
B9 LCSF–LSTM + OOF CatBoost     3.0      15647 173.958 270.667 11.597 18.044 0.562  7.417
B9 LCSF–LSTM + OOF CatBoost     6.0      15609 223.449 337.693 14.897 22.513 0.317 12.555
✓ Kaydedildi


In [ ]:
pairs = [('B6','B7','Sızıntı: Full VMD vs Rolling VMD'),
         ('B8L','B8','Sızıntı: LCSF çift yönlü vs nedensel'),
         ('B9','B1','Önerilen vs Persistence'),
         ('B9','B8','OOF katkısı (B9 vs B8)'),
         ('B9','B3','B9 vs CatBoost Direct'),
         ('B8','B7','LCSF vs Rolling VMD (nedensel)'),
         ('B7','B2','Rolling VMD vs Raw LSTM'),
         ('B8','B2','LCSF vs Raw LSTM')]
rows = []
for m1, m2, ad in pairs:
    for H in H_LIST:
        if (m1,H) in results and (m2,H) in results:
            r1, r2 = results[(m1,H)], results[(m2,H)]
            assert np.array_equal(r1['t'], r2['t'])
            y = r1['y']; r = dm_test(y, r1['p'], r2['p'], h=H, loss='abs')
            rows.append(dict(karsilastirma=ad, H_saat=H/2,
                             MAE_1=np.mean(np.abs(y-r1['p'])), MAE_2=np.mean(np.abs(y-r2['p'])),
                             DM=r['DM'], p_ham=r['p_raw']))
dm = pd.DataFrame(rows)
dm['p_holm'] = holm(dm['p_ham'].values)
dm['anlamli'] = np.where(dm['p_holm']<0.05, 'Evet', 'Hayır')
pd.set_option('display.width', 200)
print(dm.assign(DM=dm.DM.round(2), MAE_1=dm.MAE_1.round(1), MAE_2=dm.MAE_2.round(1),
                p_ham=dm.p_ham.map('{:.2e}'.format), p_holm=dm.p_holm.map('{:.2e}'.format)).to_string(index=False))
print('\nDM<0: ilk model daha iyi. Kayıp: |hata| (MAE). Holm: tablodaki tüm testler üzerinde.')
dm.to_csv(OUT+'adim5_dm_holm_T57.csv', index=False)

                       karsilastirma  H_saat  MAE_1  MAE_2     DM    p_ham   p_holm anlamli
    Sızıntı: Full VMD vs Rolling VMD     0.5   53.3   86.8 -25.61 0.00e+00 0.00e+00    Evet
    Sızıntı: Full VMD vs Rolling VMD     1.0   64.7  116.1 -26.56 0.00e+00 0.00e+00    Evet
    Sızıntı: Full VMD vs Rolling VMD     2.0   67.7  159.0 -27.76 0.00e+00 0.00e+00    Evet
    Sızıntı: Full VMD vs Rolling VMD     3.0   73.4  191.0 -26.21 0.00e+00 0.00e+00    Evet
    Sızıntı: Full VMD vs Rolling VMD     4.0   96.8  215.9 -21.50 0.00e+00 0.00e+00    Evet
    Sızıntı: Full VMD vs Rolling VMD     6.0   88.5  247.9 -22.58 0.00e+00 0.00e+00    Evet
Sızıntı: LCSF çift yönlü vs nedensel     1.0   57.3  116.3 -30.62 0.00e+00 0.00e+00    Evet
Sızıntı: LCSF çift yönlü vs nedensel     3.0   91.9  189.6 -22.13 0.00e+00 0.00e+00    Evet
Sızıntı: LCSF çift yönlü vs nedensel     6.0  116.1  243.5 -17.81 0.00e+00 0.00e+00    Evet
             Önerilen vs Persistence     1.0  108.5  108.4   0.10 9.16e-01 9.72e

In [ ]:
import time
t0 = time.time()
for _ in range(5): _ = make_lcsf(x_dec, True)
t_lcsf = (time.time()-t0)/5                       # tüm seri, tek seferde

t0 = time.time()
for i in range(20): _ = last_mode(x_dec[5000+i-W+1:5000+i+1])
t_vmd1 = (time.time()-t0)/20                       # Rolling VMD: tek zaman adımı

state = [lfilter_zi(b, a)*x_dec[0] for b, a in bands]
t0 = time.time()
for i in range(2000):
    for k, (b, a) in enumerate(bands):
        _, state[k] = lfilter(b, a, [x_dec[i]], zi=state[k])
t_lcsf1 = (time.time()-t0)/2000                    # LCSF: tek yeni örnek (çevrimiçi)

t0 = time.time(); _ = vmd_sorted(x_dec); t_full = time.time()-t0

print(f'Tüm seri ({N:,} adım) için özellik üretimi:')
print(f'  LCSF (nedensel)        : {t_lcsf:.3f} sn')
print(f'  Full VMD (sızdıran)    : {t_full:.1f} sn (tek çağrı)')
print(f'  Rolling VMD (nedensel) : ≈ {t_vmd1*N/60:.0f} dk tek çekirdekte ({t_vmd1*1000:.1f} ms × {N:,} çağrı)')
print(f'Yeni bir örnek geldiğinde (çevrimiçi) gecikme:')
print(f'  LCSF         : {t_lcsf1*1e6:.0f} µs')
print(f'  Rolling VMD  : {t_vmd1*1e3:.1f} ms  → LCSF yaklaşık {t_vmd1/t_lcsf1:.0f}× daha hızlı')

Tüm seri (35,089 adım) için özellik üretimi:
  LCSF (nedensel)        : 0.008 sn
  Full VMD (sızdıran)    : 36.4 sn (tek çağrı)
  Rolling VMD (nedensel) : ≈ 27 dk tek çekirdekte (46.3 ms × 35,089 çağrı)
Yeni bir örnek geldiğinde (çevrimiçi) gecikme:
  LCSF         : 112 µs
  Rolling VMD  : 46.3 ms  → LCSF yaklaşık 413× daha hızlı


In [ ]:
def stack(D, base, L, H, mode):
    F = lambda X, p: np.column_stack([X[:,-1,:], p])
    if mode == 'oof':
        oof = oof_predictions(D['tr'], L, H); ok = ~np.isnan(oof)
    else:                                                   # eğitim-içi artık (OOF değil)
        oof = pred_lstm(base, D['tr']['X']); ok = np.ones(len(oof), bool)
    base_va = pred_lstm(base, D['va']['X']); base_te = pred_lstm(base, D['te']['X'])
    cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, loss_function='MAE',
                           random_seed=SEED, verbose=0, early_stopping_rounds=50)
    cb.fit(F(D['tr']['X'][ok], oof[ok]), D['tr']['y'][ok]-oof[ok],
           eval_set=(F(D['va']['X'], base_va), D['va']['y']-base_va))
    return np.clip(base_te + cb.predict(F(D['te']['X'], base_te)), 0, None)

rows = []
for H in H4:
    # B8I: LCSF-LSTM + eğitim-içi artık CatBoost
    D8 = Dcache[H]
    p = stack(D8, base_models[H], L_BEST, H, 'insample')
    results[('B8I',H)] = dict(t=D8['te']['t'], y=D8['te']['y'], yc=D8['te']['yc'], p=p)
    rows.append(dict(model='B8I LCSF + eğitim-içi artık', H_saat=H/2, **metrics(D8['te']['y'], p, D8['te']['yc'])))
    # B2O: Raw LSTM (LCSF yok) + OOF CatBoost
    D2 = get_xy_ext(L_BEST, H, np.zeros((N,0)))
    assert np.array_equal(D2['te']['t'], results[('B1',H)]['t'])
    m2 = fit_lstm(D2, L_BEST)
    p = stack(D2, m2, L_BEST, H, 'oof')
    results[('B2O',H)] = dict(t=D2['te']['t'], y=D2['te']['y'], yc=D2['te']['yc'], p=p)
    rows.append(dict(model='B2O Raw LSTM + OOF CatBoost', H_saat=H/2, **metrics(D2['te']['y'], p, D2['te']['yc'])))
    print(f'H={H/2} sa tamam')

print(pd.DataFrame(rows).round(3).to_string(index=False))
pickle.dump(results, open(OUT+'adim2_T57_tahminler.pkl','wb'))

# ablasyon DM testleri (Holm: bu 6 test içinde)
ab = []
for m1, m2_, ad in [('B9','B2O','LCSF katkısı (B9 vs B2O)'), ('B9','B8I','OOF gerekli mi (B9 vs B8I)')]:
    for H in H4:
        r1, r2 = results[(m1,H)], results[(m2_,H)]; y = r1['y']
        r = dm_test(y, r1['p'], r2['p'], h=H, loss='abs')
        ab.append(dict(karsilastirma=ad, H_saat=H/2, MAE_1=np.mean(np.abs(y-r1['p'])),
                       MAE_2=np.mean(np.abs(y-r2['p'])), DM=r['DM'], p_ham=r['p_raw']))
ab = pd.DataFrame(ab); ab['p_holm'] = holm(ab['p_ham'].values)
print(ab.round(4).to_string(index=False)); print('✓ Kaydedildi')

H=1.0 sa tamam
H=3.0 sa tamam
H=6.0 sa tamam
                      model  H_saat     MAE    RMSE   nMAE  nRMSE    R2  Skill
B8I LCSF + eğitim-içi artık     1.0 106.958 181.417  7.131 12.094 0.804  1.344
B2O Raw LSTM + OOF CatBoost     1.0 108.767 184.373  7.251 12.292 0.798 -0.325
B8I LCSF + eğitim-içi artık     3.0 173.769 268.177 11.585 17.878 0.570  7.518
B2O Raw LSTM + OOF CatBoost     3.0 179.302 272.315 11.953 18.154 0.557  4.573
B8I LCSF + eğitim-içi artık     6.0 220.820 329.885 14.721 21.992 0.348 13.583
B2O Raw LSTM + OOF CatBoost     6.0 222.899 343.263 14.860 22.884 0.294 12.770
             karsilastirma  H_saat    MAE_1    MAE_2      DM  p_ham  p_holm
  LCSF katkısı (B9 vs B2O)     1.0 108.5308 108.7673 -0.5694 0.5691  1.0000
  LCSF katkısı (B9 vs B2O)     3.0 173.9577 179.3016 -5.1299 0.0000  0.0000
  LCSF katkısı (B9 vs B2O)     6.0 223.4494 222.8988  0.1614 0.8718  1.0000
OOF gerekli mi (B9 vs B8I)     1.0 108.5308 106.9579  3.1619 0.0016  0.0079
OOF gerekli mi (B9 vs 

In [ ]:
def fit_arrays(Xt, yt, Xv, yv, L):
    tf.keras.utils.set_random_seed(SEED)
    m = build_lstm(L, Xt.shape[2])
    m.fit(Xt, yt/KAPASITE, validation_data=(Xv, yv/KAPASITE), epochs=50, batch_size=256, verbose=0,
          callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])
    return m

def oof_predictions(Dtr, L, H, n_blocks=4):
    n = len(Dtr['y']); ed = np.linspace(0, n, n_blocks+1).astype(int)
    oof = np.full(n, np.nan); gap = H + L
    for j in range(1, n_blocks):
        te_end = ed[j] - gap; cut = int(te_end*0.85)
        m = fit_arrays(Dtr['X'][:cut], Dtr['y'][:cut], Dtr['X'][cut:te_end], Dtr['y'][cut:te_end], L)
        oof[ed[j]:ed[j+1]] = pred_lstm(m, Dtr['X'][ed[j]:ed[j+1]])
    return oof

def stack(D, base, L, H, mode):
    F = lambda X, p: np.column_stack([X[:,-1,:], p])
    if mode == 'oof':
        oof = oof_predictions(D['tr'], L, H); ok = ~np.isnan(oof)
    else:
        oof = pred_lstm(base, D['tr']['X']); ok = np.ones(len(oof), bool)
    base_va = pred_lstm(base, D['va']['X']); base_te = pred_lstm(base, D['te']['X'])
    cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, loss_function='MAE',
                           random_seed=SEED, verbose=0, early_stopping_rounds=50)
    cb.fit(F(D['tr']['X'][ok], oof[ok]), D['tr']['y'][ok]-oof[ok],
           eval_set=(F(D['va']['X'], base_va), D['va']['y']-base_va))
    return np.clip(base_te + cb.predict(F(D['te']['X'], base_te)), 0, None)

from joblib import Parallel, delayed
from scipy.signal import butter, lfilter, lfilter_zi, filtfilt
import os, time, pickle

def set_turbine(tid):
    global d, tr, va, te, mn, mx, rng_, sc, x_dec, N, bands
    d = add_features(load_turbine(df_all, tid)); tr, va, te = chrono_split(d, purge=PURGE)
    mn = tr[FEATURES].min().values; mx = tr[FEATURES].max().values
    rng_ = np.where(mx-mn==0, 1, mx-mn)
    sc = lambda x: (x[FEATURES].values-mn)/rng_
    x_dec = d['Patv'].ffill().fillna(0).values.astype(float); N = len(x_dec)
    _, _, om = VMD(x_dec[:len(tr)], ALPHA, TAU, K, DC, INIT, TOL)
    fc = np.clip(np.sort(om[-1]), 1e-4, 0.49); edges = np.sqrt(fc[:-1]*fc[1:])
    bands = []
    for k in range(K):
        if k == 0:       bands.append(butter(3, edges[0], 'low', fs=1.0))
        elif k == K-1:   bands.append(butter(3, edges[K-2], 'high', fs=1.0))
        else:            bands.append(butter(3, [edges[k-1], edges[k]], 'band', fs=1.0))
    return fc

def make_lcsf(x, causal=True):
    out = np.zeros((len(x), K))
    for k, (b, a) in enumerate(bands):
        if causal:
            zi = lfilter_zi(b, a)*x[0]; out[:,k] = lfilter(b, a, x, zi=zi)[0]
        else:
            out[:,k] = filtfilt(b, a, x)
    return out

def rolling_vmd(tid):
    fn_all = OUT + f'vmd_roll_T{tid}.npy'
    if os.path.exists(fn_all): return np.load(fn_all)
    U = np.full((N, K), np.nan)
    for s in range(W-1, N, 2000):
        e = min(s+2000, N); fn = OUT + f'rollvmd_T{tid}_blk_{s}.npy'
        if os.path.exists(fn): U[s:e] = np.load(fn); continue
        t0 = time.time()
        res = Parallel(n_jobs=-1)(delayed(last_mode)(x_dec[t-W+1:t+1]) for t in range(s, e))
        blk = np.nan_to_num(np.array(res)); np.save(fn, blk); U[s:e] = blk
        print(f'   rolling VMD T{tid}: {e}/{N} | {time.time()-t0:.0f} sn')
    np.save(fn_all, U); return U

def run_turbine(tid, Hs=(2,)):
    t_start = time.time(); fc = set_turbine(tid)
    print(f'T{tid}: N={N:,} | eğitim/doğrulama/test = {len(tr):,}/{len(va):,}/{len(te):,} | frekanslar={fc.round(4)}')
    fa = OUT + f'vmd_full_T{tid}.npy'
    if os.path.exists(fa): uA_ = np.load(fa)
    else:
        uA_ = vmd_sorted(x_dec).T
        if uA_.shape[0] < N: uA_ = np.vstack([uA_, np.repeat(uA_[-1:], N-uA_.shape[0], 0)])
        np.save(fa, uA_)
    uB_ = rolling_vmd(tid)
    uA_m = uA_.copy(); uA_m[:W-1] = np.nan
    cC = make_lcsf(x_dec, True); cL = make_lcsf(x_dec, False); cC[:W-1] = np.nan; cL[:W-1] = np.nan
    Ut = {'B6': scale_comp(uA_m), 'B7': scale_comp(uB_), 'B8': scale_comp(cC), 'B8L': scale_comp(cL)}
    res = {}
    for H in Hs:
        D0 = get_xy_ext(L_BEST, H, np.zeros((N,0))); t0_ = D0['te']
        put = lambda key, D, p: res.__setitem__((key,H), dict(t=D['te']['t'], y=D['te']['y'], yc=D['te']['yc'], p=p))
        put('B1', D0, t0_['yc'])
        m2 = fit_lstm(D0, L_BEST); put('B2', D0, pred_lstm(m2, t0_['X']))
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, loss_function='MAE',
                               random_seed=SEED, verbose=0, early_stopping_rounds=50)
        cb.fit(D0['tr']['X'][:,-1,:], D0['tr']['y'], eval_set=(D0['va']['X'][:,-1,:], D0['va']['y']))
        put('B3', D0, np.clip(cb.predict(t0_['X'][:,-1,:]), 0, None))
        for key in ('B6','B7','B8','B8L'):
            D = get_xy_ext(L_BEST, H, Ut[key]); assert np.array_equal(D['te']['t'], t0_['t']), 'test kümesi farklı'
            m = fit_lstm(D, L_BEST); put(key, D, pred_lstm(m, D['te']['X']))
            if key == 'B8': D8, m8 = D, m
        put('B9', D8, stack(D8, m8, L_BEST, H, 'oof'))
        print(f'   T{tid} H={H/2} sa modeller tamam | {(time.time()-t_start)/60:.1f} dk')
    pickle.dump(res, open(OUT + f'adim6_T{tid}.pkl', 'wb'))
    return res

def ozet(res, tid, Hs=(2,)):
    for H in Hs:
        ad = {'B1':'Persistence','B2':'Raw LSTM','B3':'CatBoost Direct','B6':'Full VMD-LSTM (sızdıran)',
              'B7':'Rolling VMD-LSTM (nedensel)','B8L':'LCSF çift yönlü (sızdıran)','B8':'LCSF nedensel','B9':'B9 LCSF+OOF CatBoost'}
        print(f'\nTürbin {tid} | H={H/2} sa')
        mae = {k: np.mean(np.abs(res[(k,H)]['y']-res[(k,H)]['p'])) for k in ad}
        for k in ad: print(f'  {ad[k]:32s} MAE={mae[k]:8.2f}  Skill={100*(mae["B1"]-mae[k])/mae["B1"]:7.2f}%')
        print(f'  LSI VMD  = {lsi(mae["B7"], mae["B6"]):.1f}%  |  LSI LCSF = {lsi(mae["B8"], mae["B8L"]):.1f}%')
        for m1, m2_, nm in (('B6','B7','VMD sızıntı'), ('B8L','B8','LCSF sızıntı')):
            r = dm_test(res[(m1,H)]['y'], res[(m1,H)]['p'], res[(m2_,H)]['p'], h=H)
            print(f'  DM {nm}: {r["DM"]:.2f}, p={r["p_raw"]:.2e}')

In [ ]:
res10 = run_turbine(10)
ozet(res10, 10)

T10: N=35,089 | eğitim/doğrulama/test = 24,562/5,251/5,252 | frekanslar=[0.0006 0.0187 0.0434 0.0886]
   rolling VMD T10: 2671/35089 | 196 sn
   rolling VMD T10: 4671/35089 | 186 sn
   rolling VMD T10: 6671/35089 | 124 sn
   rolling VMD T10: 8671/35089 | 212 sn
   rolling VMD T10: 10671/35089 | 263 sn
   rolling VMD T10: 12671/35089 | 229 sn
   rolling VMD T10: 14671/35089 | 253 sn
   rolling VMD T10: 16671/35089 | 295 sn
   rolling VMD T10: 18671/35089 | 267 sn
   rolling VMD T10: 20671/35089 | 224 sn
   rolling VMD T10: 22671/35089 | 223 sn
   rolling VMD T10: 24671/35089 | 291 sn
   rolling VMD T10: 26671/35089 | 196 sn
   rolling VMD T10: 28671/35089 | 245 sn
   rolling VMD T10: 30671/35089 | 248 sn
   rolling VMD T10: 32671/35089 | 231 sn
   rolling VMD T10: 34671/35089 | 245 sn
   rolling VMD T10: 35089/35089 | 70 sn
   T10 H=1.0 sa modeller tamam | 74.0 dk

Türbin 10 | H=1.0 sa
  Persistence                      MAE=  110.58  Skill=   0.00%
  Raw LSTM                         MAE

In [ ]:
res83 = run_turbine(83)
ozet(res83, 83)

T83: N=35,089 | eğitim/doğrulama/test = 24,562/5,251/5,252 | frekanslar=[0.0009 0.0186 0.0461 0.096 ]
   rolling VMD T83: 2671/35089 | 131 sn
   rolling VMD T83: 4671/35089 | 8 sn
   rolling VMD T83: 6671/35089 | 83 sn
   rolling VMD T83: 8671/35089 | 180 sn
   rolling VMD T83: 10671/35089 | 267 sn
   rolling VMD T83: 12671/35089 | 243 sn
   rolling VMD T83: 14671/35089 | 242 sn
   rolling VMD T83: 16671/35089 | 321 sn
   rolling VMD T83: 18671/35089 | 269 sn
   rolling VMD T83: 20671/35089 | 221 sn
   rolling VMD T83: 22671/35089 | 202 sn
   rolling VMD T83: 24671/35089 | 240 sn
   rolling VMD T83: 26671/35089 | 227 sn
   rolling VMD T83: 28671/35089 | 215 sn
   rolling VMD T83: 30671/35089 | 240 sn
   rolling VMD T83: 32671/35089 | 359 sn
   rolling VMD T83: 34671/35089 | 221 sn
   rolling VMD T83: 35089/35089 | 79 sn
   T83 H=1.0 sa modeller tamam | 69.8 dk

Türbin 83 | H=1.0 sa
  Persistence                      MAE=   95.85  Skill=   0.00%
  Raw LSTM                         MAE=  

In [ ]:
res120 = run_turbine(120)
ozet(res120, 120)

T120: N=35,089 | eğitim/doğrulama/test = 24,562/5,251/5,252 | frekanslar=[0.0008 0.0192 0.0445 0.0933]
   rolling VMD T120: 2671/35089 | 138 sn
   rolling VMD T120: 4671/35089 | 246 sn
   rolling VMD T120: 6671/35089 | 124 sn
   rolling VMD T120: 8671/35089 | 210 sn
   rolling VMD T120: 10671/35089 | 271 sn
   rolling VMD T120: 12671/35089 | 251 sn
   rolling VMD T120: 14671/35089 | 229 sn
   rolling VMD T120: 16671/35089 | 323 sn
   rolling VMD T120: 18671/35089 | 260 sn
   rolling VMD T120: 20671/35089 | 181 sn
   rolling VMD T120: 22671/35089 | 221 sn
   rolling VMD T120: 24671/35089 | 245 sn
   rolling VMD T120: 26671/35089 | 204 sn
   rolling VMD T120: 28671/35089 | 259 sn
   rolling VMD T120: 30671/35089 | 220 sn
   rolling VMD T120: 32671/35089 | 287 sn
   rolling VMD T120: 34671/35089 | 220 sn
   rolling VMD T120: 35089/35089 | 76 sn
   T120 H=1.0 sa modeller tamam | 72.8 dk

Türbin 120 | H=1.0 sa
  Persistence                      MAE=   95.08  Skill=   0.00%
  Raw LSTM       